# CinnaAI — Cinnamon Bark Disease Training in Google Colab

**Classes, in exact output order:** `RoughBark`, `StripeCanker`. **Healthy bark is not supported.**

Select **Runtime → Change runtime type → GPU**, then run the numbered sections from top to bottom.
CELL 2 installs only missing packages and retains the current Colab TensorFlow/Keras stack.
The notebook downloads the supplied Kaggle dataset, uses ImageNet EfficientNetB0, trains the frozen
classifier head followed by selective fine-tuning, selects a stage using validation macro-F1,
chooses uncertainty thresholds on validation data, and only then measures held-out test performance.
The final ZIP includes model exports, environment records, metrics, hashes and real-image parity fixtures.
Optional manual upload is disabled by default so **Run all** can finish.

## Reconstruction notice

The supplied text contained collapsed Python statements and incomplete cells. The user authorized
restoring formatting and reconstructing missing code. All **97 supplied CELL titles and their order**
are retained. Each section states its repairs; the original supplied section is preserved in its code
cell metadata (`cinnaai_original_section`). The repair log at the end documents the material choices.

The split remains the supplied **image-level 70/15/15 stratified split**, not a claimed tree/site-independent
evaluation. All copies of exact files with conflicting labels are excluded and audited before splitting. Same-label exact file duplicates are deduplicated; near-duplicate pairs are reported only.
If the validation accuracy/coverage targets cannot be met, automatic acceptance is disabled.
Both formats are exported, but a failed parity result is recorded and the artifact is **diagnostic only**.
Numerical compatibility with the real CinnaAI backend still needs its own test after Colab training.

### Colab setup and references

- [Colab GPU/runtime guidance](https://research.google.com/colaboratory/faq.html).
- [KaggleHub dataset download and authentication](https://github.com/Kaggle/kagglehub).
  If the dataset requires consent or access credentials, grant access in Kaggle and use its documented
  Colab secret/login setup before retrying CELL 6; never paste tokens into a shared notebook.
- [Keras fine-tuning and BatchNormalization](https://keras.io/guides/transfer_learning/).
- [Keras Grad-CAM example](https://keras.io/examples/vision/grad_cam/).

**Before CELL 21 or CELL 64 is rerun:** their supplied cleanup removes their previous `/content`
working/handoff directory. Download the prior handoff first if you need to retain it.


## CELL 1 — Check GPU

**Repair record:** Preserved the supplied Colab shell command.


In [ ]:
!nvidia-smi


## CELL 2 — Install only missing packages

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
import sys
import subprocess
import importlib.util

REQUIRED_PACKAGES = {
    "kagglehub": "kagglehub", "tensorflow": "tensorflow", "keras": "keras",
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
    "sklearn": "scikit-learn", "PIL": "Pillow", "h5py": "h5py",
}
missing_packages = [pip_name for module_name, pip_name in REQUIRED_PACKAGES.items()
                    if importlib.util.find_spec(module_name) is None]
if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
else:
    print("All required packages are already available. Keeping the current Colab stack.")


## CELL 3 — Imports

**Repair record:** Restored imports; added importlib.metadata to record installed package versions without changing packages.


In [ ]:
import os
import sys
import json
import math
import shutil
import hashlib
import random
import platform
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata as package_metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display
import tensorflow as tf
import keras
import sklearn
import PIL
import h5py
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_class_weight


## CELL 4 — Print and record Colab environment

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
print("=" * 70)
print("TRAINING ENVIRONMENT")
print("=" * 70)
for name, version in {
    "Python": platform.python_version(), "TensorFlow": tf.__version__,
    "Keras": keras.__version__, "NumPy": np.__version__, "Pandas": pd.__version__,
    "Scikit-learn": sklearn.__version__, "Pillow": PIL.__version__, "h5py": h5py.__version__,
}.items():
    print(f"{name:14s}: {version}")
print("GPU devices:", tf.config.list_physical_devices("GPU"))
if not tf.config.list_physical_devices("GPU"):
    print("WARNING: TensorFlow cannot see a GPU. Select a GPU runtime before training.")
else:
    print("PASS: TensorFlow GPU detected.")


## CELL 5 — Reproducibility settings

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
    print("Deterministic TensorFlow operations enabled.")
except Exception as e:
    print("Could not fully enable deterministic mode:", e)


## CELL 6 — Download dataset from Kaggle

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
KAGGLE_DATASET_ID = (
    "madhavipethangoda/"
    "cinnamon-plant-stem-and-branch-disease-dataset"
)
dataset_path = kagglehub.dataset_download(KAGGLE_DATASET_ID)
DATASET_ROOT = Path(dataset_path)
print("Dataset downloaded to:", DATASET_ROOT)


## CELL 7 — Display dataset structure

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
print("Dataset root:", DATASET_ROOT)
print("Top-level contents:")
for item in sorted(DATASET_ROOT.iterdir()):
    item_type = "DIR" if item.is_dir() else "FILE"
    print(f"[{item_type}] {item.name}")


## CELL 8 — Automatically locate the Diseases folder

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
disease_folder_candidates = []
for candidate in DATASET_ROOT.rglob("Diseases"):
    if candidate.is_dir() and (candidate / "RoughBark").is_dir() and (candidate / "StripeCanker").is_dir():
        disease_folder_candidates.append(candidate)
disease_folder_candidates = sorted(disease_folder_candidates)
print("Matching Diseases folders:")
for candidate in disease_folder_candidates:
    print(candidate)
if not disease_folder_candidates:
    raise FileNotFoundError("Could not locate a Diseases folder containing both RoughBark and StripeCanker.")
DATASET_DIR = disease_folder_candidates[0]
print("Selected dataset directory:", DATASET_DIR)


## CELL 9 — Define the exact model class order

`output[0] = RoughBark`; `output[1] = StripeCanker`. Never change this order after training.

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
CLASS_NAMES = ["RoughBark", "StripeCanker"]
CLASS_TO_ID = {class_name: class_id for class_id, class_name in enumerate(CLASS_NAMES)}
CLASS_DIRS = {class_name: DATASET_DIR / class_name for class_name in CLASS_NAMES}
print("Class mapping:", CLASS_TO_ID)
for class_name, class_path in CLASS_DIRS.items():
    print(class_name, "->", class_path, "| Exists:", class_path.is_dir())
    if not class_path.is_dir():
        raise FileNotFoundError(class_path)


## CELL 10 — Build image manifest

**Repair record:** Completed the class-count display and supplied explicit columns so an empty manifest receives a clear validation error.


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
records = []
for class_id, class_name in enumerate(CLASS_NAMES):
    for image_path in sorted(CLASS_DIRS[class_name].rglob("*")):
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
            records.append({"path": str(image_path), "filename": image_path.name,
                            "class_id": class_id, "class_name": class_name})
df = pd.DataFrame(records, columns=["path", "filename", "class_id", "class_name"])
print("Total images found:", len(df))
print("Images per class:")
display(df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0))


## CELL 11 — Make sure both classes contain enough images

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
class_counts = df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0)
print(class_counts)
for class_name in CLASS_NAMES:
    count = int(class_counts[class_name])
    if count < 20:
        raise RuntimeError(f"{class_name} has only {count} images. This is too small for the planned split.")
print("PASS: both classes contain enough images for the planned split; this is not a field-validity guarantee.")


## CELL 12 — Validate every image

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
def validate_image(image_path):
    try:
        with Image.open(image_path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            image.load()
        return True, None
    except Exception as e:
        return False, str(e)

validation_records = []
for _, row in df.iterrows():
    valid, error = validate_image(row["path"])
    validation_records.append({"path": row["path"], "valid": valid, "error": error})
image_validation_df = pd.DataFrame(validation_records)
bad_images = image_validation_df[~image_validation_df["valid"]].copy()
print("Invalid images:", len(bad_images))
if not bad_images.empty:
    display(bad_images)


## CELL 13 — Remove invalid images

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
valid_paths = set(image_validation_df.loc[image_validation_df["valid"], "path"])
df = df[df["path"].isin(valid_paths)].reset_index(drop=True)
print("Usable images:", len(df))
print(df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0))


## CELL 14 — Calculate SHA-256 hashes

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
def sha256_file(file_path):
    hasher = hashlib.sha256()
    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

df["sha256"] = [sha256_file(file_path) for file_path in df["path"]]
print("SHA-256 hashes calculated.")


## CELL 15 — Find exact duplicates

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
duplicate_rows = df.groupby("sha256").filter(lambda group: len(group) > 1).sort_values("sha256")
print("Rows involved in exact duplicates:", len(duplicate_rows))
if len(duplicate_rows):
    display(duplicate_rows[["filename", "class_name", "path", "sha256"]])


## CELL 16 — Detect duplicate images with conflicting labels

All copies of an identical file with contradictory disease labels are excluded from the manifest. The original dataset files are retained. No folder-order label is assumed.

**Repair record:** Exclude all copies of cross-label identical files from the in-memory manifest, preserve originals, and retain an audit table instead of stopping or selecting a label arbitrarily.


In [ ]:
duplicate_class_counts = df.groupby("sha256")["class_name"].nunique()
conflicting_hashes = duplicate_class_counts[duplicate_class_counts > 1].index.tolist()
conflicting_images = df[df["sha256"].isin(conflicting_hashes)].sort_values("sha256").copy()
CONFLICTING_HASH_GROUPS_REMOVED = len(conflicting_hashes)
CONFLICTING_IMAGES_REMOVED = len(conflicting_images)
print("Identical files with conflicting labels:", CONFLICTING_HASH_GROUPS_REMOVED)
if conflicting_hashes:
    display(conflicting_images)
    # Exclude every copy: folder order cannot establish the correct disease label.
    df = df[~df["sha256"].isin(conflicting_hashes)].reset_index(drop=True)
    print("Excluded all conflicting copies from training/evaluation:", CONFLICTING_IMAGES_REMOVED)
    print("Original files are untouched; excluded rows will be saved in the handoff audit.")
else:
    print("PASS: no conflicting exact duplicates.")
assert not (df.groupby("sha256")["class_name"].nunique() > 1).any()
print("Remaining class counts:")
print(df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0))


## CELL 17 — Remove duplicate copies

Only same-label exact duplicates may be reduced to one copy. CELL 16 must exclude cross-label conflicts first.

**Repair record:** Completed duplicate removal and post-cleaning class checks; added a pre-deduplication guard so CELL 17 cannot silently choose a label for an unresolved cross-label file.


In [ ]:
assert not (df.groupby("sha256")["class_name"].nunique() > 1).any(), "Run CELL 16 before deduplicating."
before_deduplication = len(df)
df = (df.sort_values(["class_id", "path"]).drop_duplicates(subset=["sha256"], keep="first")
      .reset_index(drop=True))
after_deduplication = len(df)
DUPLICATES_REMOVED = before_deduplication - after_deduplication
print("Images before:", before_deduplication)
print("Images after:", after_deduplication)
print("Exact duplicates removed:", DUPLICATES_REMOVED)
for class_name in CLASS_NAMES:
    count = int((df["class_name"] == class_name).sum())
    if count < 20:
        raise RuntimeError(f"After cleaning, {class_name} has only {count} images; collect/review data before splitting.")


## CELL 18 — Near-duplicate audit

This audit only reports suspicious pairs. Review them; no images are automatically deleted or grouped.

**Repair record:** The source only defined a hash function. Added candidate comparison, a results table, and explicit resource-cap/completeness reporting; no automatic deletion.


In [ ]:
def difference_hash(image_path, hash_size=8):
    with Image.open(image_path) as image:
        image = ImageOps.exif_transpose(image).convert("L")
        image = image.resize((hash_size + 1, hash_size), Image.Resampling.BILINEAR)
        pixels = np.asarray(image)
    differences = pixels[:, 1:] > pixels[:, :-1]
    hash_value = 0
    for index, value in enumerate(differences.flatten()):
        if value:
            hash_value |= 1 << index
    return hash_value

df["dhash"] = [f"{difference_hash(path):016x}" for path in df["path"]]
# Five disjoint bit chunks find candidate pairs at Hamming distance <= 4.
# No files are removed or relabelled by this heuristic audit.
NEAR_DUPLICATE_MAX_DISTANCE = 4
MAX_AUDIT_COMPARISONS = 2_000_000
MAX_AUDIT_RESULTS = 10_000
buckets = {}
near_duplicate_records = []
audit_comparisons = 0
near_duplicate_audit_complete = True
hash_values = [int(value, 16) for value in df["dhash"]]
chunks = [(0, 13), (13, 13), (26, 13), (39, 13), (52, 12)]
for right_index, right_hash in enumerate(hash_values):
    keys = [(part, (right_hash >> offset) & ((1 << width) - 1))
            for part, (offset, width) in enumerate(chunks)]
    candidates = set()
    for key in keys:
        candidates.update(buckets.get(key, []))
    for left_index in sorted(candidates):
        if audit_comparisons >= MAX_AUDIT_COMPARISONS or len(near_duplicate_records) >= MAX_AUDIT_RESULTS:
            near_duplicate_audit_complete = False
            break
        audit_comparisons += 1
        distance = (hash_values[left_index] ^ right_hash).bit_count()
        if distance <= NEAR_DUPLICATE_MAX_DISTANCE:
            near_duplicate_records.append({
                "path_a": df.iloc[left_index]["path"], "path_b": df.iloc[right_index]["path"],
                "class_a": df.iloc[left_index]["class_name"], "class_b": df.iloc[right_index]["class_name"],
                "hamming_distance": distance,
            })
    if not near_duplicate_audit_complete:
        break
    for key in keys:
        buckets.setdefault(key, []).append(right_index)
near_duplicate_df = pd.DataFrame(near_duplicate_records,
    columns=["path_a", "path_b", "class_a", "class_b", "hamming_distance"])
print("Potential near-duplicate pairs:", len(near_duplicate_df))
print("Heuristic audit complete:", near_duplicate_audit_complete)
if not near_duplicate_df.empty:
    display(near_duplicate_df.head(30))
    print("Review these pairs. They are not automatically removed or grouped; split leakage may remain.")
if not near_duplicate_audit_complete:
    print("Audit hit its resource cap; recorded pairs are a lower bound, not a complete count.")


## CELL 19 — Plot class distribution

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
class_counts = df["class_name"].value_counts().reindex(CLASS_NAMES)
display(class_counts.rename("image_count"))
plt.figure(figsize=(7, 5))
class_counts.plot(kind="bar")
plt.title("Cinnamon Bark Disease Class Distribution")
plt.xlabel("Disease class")
plt.ylabel("Number of images")
plt.xticks(rotation=0)
plt.show()


## CELL 20 — Display random images from each class

Inspect the image labels manually. Stop if visible label errors need correction.

**Repair record:** Completed the truncated image display loop and hid unused axes.


In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 5, figsize=(16, 7), squeeze=False)
for class_index, class_name in enumerate(CLASS_NAMES):
    class_df = df[df["class_name"] == class_name]
    samples = class_df.sample(n=min(5, len(class_df)), random_state=SEED)
    for column_index, (_, row) in enumerate(samples.iterrows()):
        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            axes[class_index, column_index].imshow(image)
        axes[class_index, column_index].set_title(class_name)
    for axis in axes[class_index]:
        axis.axis("off")
plt.tight_layout()
plt.show()


## CELL 21 — Create working directory

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
WORK_DIR = Path("/content/bark_training")
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Working directory:", WORK_DIR)


## CELL 22 — Create 70/15/15 train-validation-test split

Tree/site IDs are unavailable to this workflow. Related images can remain across partitions despite exact-file deduplication.

**Repair record:** Completed test partition reporting; retained the supplied image-level split and its independence limitation.


In [ ]:
train_df, temporary_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["class_id"])
validation_df, test_df = train_test_split(temporary_df, test_size=0.50, random_state=SEED,
                                        stratify=temporary_df["class_id"])
train_df, validation_df, test_df = train_df.copy(), validation_df.copy(), test_df.copy()
train_df["split"], validation_df["split"], test_df["split"] = "train", "validation", "test"
for name, partition in [("TRAIN", train_df), ("VALIDATION", validation_df), ("TEST", test_df)]:
    print(name, "images:", len(partition))
    print(partition["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0))
    assert set(partition["class_id"]) == {0, 1}, f"Missing class in {name}"
print("Image-level stratified split. Tree/site independence has not been established.")


## CELL 23 — Verify no exact duplicate leakage

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
train_hashes = set(train_df["sha256"])
validation_hashes = set(validation_df["sha256"])
test_hashes = set(test_df["sha256"])
assert train_hashes.isdisjoint(validation_hashes)
assert train_hashes.isdisjoint(test_hashes)
assert validation_hashes.isdisjoint(test_hashes)
print("PASS: no exact duplicate file hashes cross partitions.")
print("This check does not exclude differently encoded or related images.")


## CELL 24 — Save permanent split manifest

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
split_manifest = pd.concat([train_df, validation_df, test_df], ignore_index=True)
SPLIT_MANIFEST_PATH = WORK_DIR / "split_manifest.csv"
split_manifest.to_csv(SPLIT_MANIFEST_PATH, index=False)
print("Saved:", SPLIT_MANIFEST_PATH)


## CELL 25 — Define exact bark preprocessing

EXIF correction → RGB → 224×224 Pillow bilinear → float32 `[0,255]`. **No external `/255`.**

**Repair record:** Completed the missing tf_load_image body with static shape and label dtype; preserved EXIF/RGB/Pillow bilinear/[0,255].


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16

def load_image_pillow(path_tensor):
    path = path_tensor.numpy().decode("utf-8")
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        image = image.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
        array = np.asarray(image, dtype=np.float32)
    return array

def tf_load_image(path, label):
    image = tf.py_function(load_image_pillow, [path], Tout=tf.float32)
    image.set_shape((IMG_SIZE, IMG_SIZE, 3))
    return image, tf.cast(label, tf.int32)


## CELL 26 — Define training-only augmentation

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip(mode="horizontal", seed=SEED),
    tf.keras.layers.RandomRotation(factor=0.04, fill_mode="reflect", seed=SEED),
    tf.keras.layers.RandomZoom(height_factor=(-0.08, 0.08), width_factor=(-0.08, 0.08),
                               fill_mode="reflect", seed=SEED),
    tf.keras.layers.RandomTranslation(height_factor=0.05, width_factor=0.05,
                                      fill_mode="reflect", seed=SEED),
    tf.keras.layers.RandomContrast(factor=0.10, seed=SEED),
], name="training_augmentation")


## CELL 27 — Build TensorFlow datasets

**Repair record:** Completed batching, training-only shuffle/augmentation and all three dataset definitions. Used disk cache and clipped augmented pixels to [0,255].


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(dataframe, training=False):
    paths = dataframe["path"].astype(str).to_numpy()
    labels = dataframe["class_id"].astype(np.int32).to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.map(tf_load_image, num_parallel_calls=AUTOTUNE)
    # Disk caching keeps decoded float32 images out of Colab RAM.
    cache_key = hashlib.sha256("\n".join(paths).encode("utf-8")).hexdigest()[:16]
    dataset = dataset.cache(str(WORK_DIR / f"decoded_{cache_key}"))
    if training:
        dataset = dataset.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.batch(BATCH_SIZE)
    if training:
        dataset = dataset.map(
            lambda images, labels: (tf.clip_by_value(augmentation(images, training=True), 0.0, 255.0), labels),
            num_parallel_calls=AUTOTUNE,
        )
    return dataset.prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
validation_ds = make_dataset(validation_df)
test_ds = make_dataset(test_df)
print("Training, validation and test pipelines created; test pixels have not been evaluated.")


## CELL 28 — Verify model input tensor

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
images, labels = next(iter(train_ds))
print("Batch shape:", images.shape, "dtype:", images.dtype)
print("Minimum pixel:", float(tf.reduce_min(images)))
print("Maximum pixel:", float(tf.reduce_max(images)))
assert tuple(images.shape[1:]) == (224, 224, 3)
assert images.dtype == tf.float32
assert float(tf.reduce_min(images)) >= 0.0 and float(tf.reduce_max(images)) <= 255.0


## CELL 29 — Display augmented training samples

Inspect these images to ensure augmentation preserves the labelled visual evidence.

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
sample_images, sample_labels = next(iter(train_ds))
plt.figure(figsize=(15, 8))
for index in range(min(10, len(sample_images))):
    plt.subplot(2, 5, index + 1)
    plt.imshow(np.clip(sample_images[index].numpy(), 0, 255).astype(np.uint8))
    plt.title(CLASS_NAMES[int(sample_labels[index])])
    plt.axis("off")
plt.tight_layout()
plt.show()


## CELL 30 — Calculate class weights

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
training_class_ids = np.array(sorted(train_df["class_id"].unique()))
calculated_weights = compute_class_weight(class_weight="balanced", classes=training_class_ids,
                                          y=train_df["class_id"].values)
class_weight = {int(class_id): float(weight) for class_id, weight in zip(training_class_ids, calculated_weights)}
print("Class weights:")
for class_id, weight in class_weight.items():
    print(CLASS_NAMES[class_id], "->", weight)


## CELL 31 — Build EfficientNetB0 model

**Repair record:** Completed the missing pooling/dropout/two-output softmax head and model construction. No augmentation or external rescaling in the inference graph.


In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32, name="bark_image")
backbone = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3),
)
BACKBONE_NAME = backbone.name
backbone.trainable = False
x = backbone(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = tf.keras.layers.Dropout(0.30, name="classification_dropout")(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax", dtype="float32",
                                name="bark_predictions")(x)
model = tf.keras.Model(inputs, outputs, name="cinnamon_bark_disease_model")
model.summary()


## CELL 32 — Verify model input/output contract

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("Output dtype:", model.output.dtype)
assert model.input_shape == (None, 224, 224, 3)
assert model.output_shape == (None, 2)
assert model.output.dtype == "float32"
assert CLASS_NAMES == ["RoughBark", "StripeCanker"]
print("PASS: CinnaAI model contract satisfied.")


## CELL 33 — Compile Stage 1

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)


## CELL 34 — Stage 1 callbacks

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
STAGE1_PATH = WORK_DIR / "stage1_best.keras"
stage1_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(str(STAGE1_PATH), monitor="val_loss", mode="min",
                                       save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=6,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", mode="min", factor=0.3,
                                        patience=2, min_lr=1e-6, verbose=1),
]


## CELL 35 — Train Stage 1

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
history_stage1 = model.fit(train_ds, validation_data=validation_ds, epochs=25,
                           class_weight=class_weight, callbacks=stage1_callbacks, verbose=1)


## CELL 36 — Prediction helper

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
def get_predictions(trained_model, dataframe, dataset):
    probabilities = np.asarray(trained_model.predict(dataset, verbose=0), dtype=np.float32)
    assert probabilities.shape == (len(dataframe), len(CLASS_NAMES))
    assert np.isfinite(probabilities).all()
    assert (probabilities >= 0.0).all() and (probabilities <= 1.0).all()
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0, atol=1e-5, rtol=0)
    predicted_ids = np.argmax(probabilities, axis=1)
    true_ids = dataframe["class_id"].to_numpy()
    return true_ids, predicted_ids, probabilities


## CELL 37 — Load and evaluate best Stage 1 checkpoint

**Repair record:** Completed the truncated validation output, retaining the best validation-loss checkpoint and recording all comparison metrics.


In [ ]:
stage1_model = tf.keras.models.load_model(STAGE1_PATH, compile=False)
y_validation, stage1_predictions, stage1_probabilities = get_predictions(stage1_model, validation_df, validation_ds)
stage1_accuracy = accuracy_score(y_validation, stage1_predictions)
stage1_macro_f1 = f1_score(y_validation, stage1_predictions, average="macro", zero_division=0)
stage1_balanced_accuracy = balanced_accuracy_score(y_validation, stage1_predictions)
print("Stage 1 validation accuracy:", stage1_accuracy)
print("Stage 1 validation macro-F1:", stage1_macro_f1)
print("Stage 1 validation balanced accuracy:", stage1_balanced_accuracy)


## CELL 38 — Start Stage 2 from the BEST Stage 1 model

Stage 2 starts from a separate copy of the best Stage 1 checkpoint. Backbone BatchNormalization stays frozen.

**Repair record:** Fixed the supplied alias model=stage1_model: reload an independent checkpoint before fine-tuning so Stage 1 remains selectable. Explicitly set both frozen and unfrozen flags.


In [ ]:
# Load a separate instance: Stage 2 must not mutate the Stage 1 candidate.
model = tf.keras.models.load_model(STAGE1_PATH, compile=False)
backbone = model.get_layer(BACKBONE_NAME)
backbone.trainable = True
FINE_TUNE_FRACTION = 0.25
fine_tune_from = int(len(backbone.layers) * (1.0 - FINE_TUNE_FRACTION))
for layer_index, layer in enumerate(backbone.layers):
    layer.trainable = (layer_index >= fine_tune_from
                       and not isinstance(layer, tf.keras.layers.BatchNormalization))
print("Trainable backbone layers:", sum(layer.trainable for layer in backbone.layers))
assert not any(layer.trainable for layer in backbone.layers
               if isinstance(layer, tf.keras.layers.BatchNormalization))


## CELL 39 — Recompile Stage 2

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)


## CELL 40 — Stage 2 callbacks

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
STAGE2_PATH = WORK_DIR / "stage2_best.keras"
stage2_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(str(STAGE2_PATH), monitor="val_loss", mode="min",
                                       save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=7,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", mode="min", factor=0.3,
                                        patience=2, min_lr=1e-7, verbose=1),
]


## CELL 41 — Train Stage 2

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
history_stage2 = model.fit(train_ds, validation_data=validation_ds, epochs=35,
                           class_weight=class_weight, callbacks=stage2_callbacks, verbose=1)


## CELL 42 — Evaluate Stage 2

**Repair record:** Completed the truncated validation output using the best Stage 2 checkpoint.


In [ ]:
stage2_model = tf.keras.models.load_model(STAGE2_PATH, compile=False)
_, stage2_predictions, stage2_probabilities = get_predictions(stage2_model, validation_df, validation_ds)
stage2_accuracy = accuracy_score(y_validation, stage2_predictions)
stage2_macro_f1 = f1_score(y_validation, stage2_predictions, average="macro", zero_division=0)
stage2_balanced_accuracy = balanced_accuracy_score(y_validation, stage2_predictions)
print("Stage 2 validation accuracy:", stage2_accuracy)
print("Stage 2 validation macro-F1:", stage2_macro_f1)
print("Stage 2 validation balanced accuracy:", stage2_balanced_accuracy)


## CELL 43 — Compare both training stages

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
comparison_df = pd.DataFrame([
    {"model": "Stage 1 - Frozen EfficientNet", "validation_accuracy": stage1_accuracy,
     "validation_macro_f1": stage1_macro_f1, "validation_balanced_accuracy": stage1_balanced_accuracy},
    {"model": "Stage 2 - Fine-tuned EfficientNet", "validation_accuracy": stage2_accuracy,
     "validation_macro_f1": stage2_macro_f1, "validation_balanced_accuracy": stage2_balanced_accuracy},
])
display(comparison_df)


## CELL 44 — Automatically select the better model

Model selection uses validation macro-F1; the test set does not select a stage.

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
if stage2_macro_f1 >= stage1_macro_f1:
    best_model = stage2_model
    chosen_stage = "stage2_fine_tuned"
    validation_probabilities = stage2_probabilities
else:
    best_model = stage1_model
    chosen_stage = "stage1_frozen"
    validation_probabilities = stage1_probabilities
print("SELECTED MODEL:", chosen_stage)
print("Selection uses validation macro-F1 only; ties retain the supplied Stage 2 preference.")


## CELL 45 — Plot Stage 1 accuracy

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history_stage1.history["accuracy"], label="Training accuracy")
plt.plot(history_stage1.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Stage 1 Accuracy")
plt.legend()
plt.show()


## CELL 46 — Plot Stage 1 loss

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history_stage1.history["loss"], label="Training loss")
plt.plot(history_stage1.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Stage 1 Loss")
plt.legend()
plt.show()


## CELL 47 — Plot Stage 2 accuracy

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history_stage2.history["accuracy"], label="Training accuracy")
plt.plot(history_stage2.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Stage 2 Accuracy")
plt.legend()
plt.show()


## CELL 48 — Plot Stage 2 loss

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history_stage2.history["loss"], label="Training loss")
plt.plot(history_stage2.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Stage 2 Loss")
plt.legend()
plt.show()


## CELL 49 — Combine and save training history

**Repair record:** Closed the truncated training-history export call.


In [ ]:
history_stage1_df = pd.DataFrame(history_stage1.history)
history_stage1_df["stage"] = "stage1"
history_stage1_df["epoch"] = np.arange(1, len(history_stage1_df) + 1)
history_stage2_df = pd.DataFrame(history_stage2.history)
history_stage2_df["stage"] = "stage2"
history_stage2_df["epoch"] = np.arange(1, len(history_stage2_df) + 1)
training_history = pd.concat([history_stage1_df, history_stage2_df], ignore_index=True)
training_history.to_csv(WORK_DIR / "training_history.csv", index=False)
display(training_history.tail())


## CELL 50 — Calculate validation confidence and margin

Confidence and margin support an uncertain state between the two disease labels; they do not establish healthy detection.

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
validation_predictions = np.argmax(validation_probabilities, axis=1)
sorted_validation_scores = np.sort(validation_probabilities, axis=1)
validation_confidence = sorted_validation_scores[:, -1]
validation_margin = sorted_validation_scores[:, -1] - sorted_validation_scores[:, -2]
validation_correct = validation_predictions == y_validation


## CELL 51 — Search validation thresholds

**Repair record:** Completed the truncated threshold search, accepted-accuracy/coverage calculations and missing threshold_df definition.


In [ ]:
threshold_results = []
for confidence_threshold in np.arange(0.50, 0.96, 0.02):
    for margin_threshold in np.arange(0.00, 0.51, 0.02):
        accepted = ((validation_confidence >= confidence_threshold)
                    & (validation_margin >= margin_threshold))
        accepted_count = int(accepted.sum())
        if accepted_count == 0:
            continue
        threshold_results.append({
            "confidence_threshold": float(confidence_threshold),
            "margin_threshold": float(margin_threshold),
            "accepted_count": accepted_count,
            "coverage": float(accepted.mean()),
            "accepted_accuracy": float(validation_correct[accepted].mean()),
        })
threshold_df = pd.DataFrame(threshold_results)
if threshold_df.empty:
    raise RuntimeError("No validation examples met even the lowest score thresholds. Inspect model outputs.")
display(threshold_df.sort_values(["coverage", "accepted_accuracy"], ascending=False).head(15))


## CELL 52 — Automatically choose uncertainty threshold

Targets are 90% accepted validation accuracy and 60% validation coverage. They are goals, not guaranteed performance.

**Repair record:** Completed threshold selection and scalar definitions. Documented a conservative missing-target fallback: disable acceptance instead of claiming unmet 90%/60% goals.


In [ ]:
TARGET_ACCEPTED_ACCURACY = 0.90
MIN_VALIDATION_COVERAGE = 0.60
eligible_thresholds = threshold_df[
    (threshold_df["accepted_accuracy"] >= TARGET_ACCEPTED_ACCURACY)
    & (threshold_df["coverage"] >= MIN_VALIDATION_COVERAGE)
].copy()
ACCEPTANCE_ENABLED = not eligible_thresholds.empty
if ACCEPTANCE_ENABLED:
    selected_threshold = eligible_thresholds.sort_values(
        ["coverage", "accepted_accuracy", "confidence_threshold", "margin_threshold"],
        ascending=[False, False, True, True], kind="stable",
    ).iloc[0]
else:
    selected_threshold = threshold_df.sort_values(
        ["accepted_accuracy", "coverage"], ascending=False, kind="stable",
    ).iloc[0]
    print("Validation targets were not met. Automatic accepted disease results are disabled; results remain uncertain.")
CONFIDENCE_THRESHOLD = float(selected_threshold["confidence_threshold"])
MARGIN_THRESHOLD = float(selected_threshold["margin_threshold"])
print("Confidence threshold:", CONFIDENCE_THRESHOLD)
print("Margin threshold:", MARGIN_THRESHOLD)
print("Automatic acceptance enabled:", ACCEPTANCE_ENABLED)
display(selected_threshold.to_frame("selected_validation_policy"))


## CELL 53 — FINAL held-out test evaluation

This is the first test-performance measurement. Do not use these results to tune the model or thresholds.

**Repair record:** Completed the truncated held-out evaluation and per-class classification report; no healthy-class metrics are invented.


In [ ]:
y_test, test_predictions, test_probabilities = get_predictions(best_model, test_df, test_ds)
test_accuracy = accuracy_score(y_test, test_predictions)
test_macro_f1 = f1_score(y_test, test_predictions, average="macro", zero_division=0)
test_balanced_accuracy = balanced_accuracy_score(y_test, test_predictions)
print("=" * 70)
print("TEST ACCURACY:", round(test_accuracy, 4))
print("TEST MACRO-F1:", round(test_macro_f1, 4))
print("TEST BALANCED ACCURACY:", round(test_balanced_accuracy, 4))
print(classification_report(y_test, test_predictions, labels=[0, 1], target_names=CLASS_NAMES, zero_division=0))
print("These are two-disease metrics; healthy/unhealthy accuracy is not established.")


## CELL 54 — Test confusion matrix

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
confusion = confusion_matrix(y_test, test_predictions, labels=[0, 1])
display_object = ConfusionMatrixDisplay(confusion_matrix=confusion, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(7, 7))
display_object.plot(ax=ax, values_format="d")
plt.title("Cinnamon Bark Disease Test Confusion Matrix")
plt.show()


## CELL 55 — Apply uncertainty policy to test results

**Repair record:** Completed truncated accepted-test metrics and handled zero accepted examples without NaN or division errors.


In [ ]:
sorted_test_scores = np.sort(test_probabilities, axis=1)
test_confidence = sorted_test_scores[:, -1]
test_margin = sorted_test_scores[:, -1] - sorted_test_scores[:, -2]
test_accepted = (ACCEPTANCE_ENABLED & (test_confidence >= CONFIDENCE_THRESHOLD)
                 & (test_margin >= MARGIN_THRESHOLD))
test_coverage = float(test_accepted.mean())
accepted_test_accuracy = (float((test_predictions[test_accepted] == y_test[test_accepted]).mean())
                          if test_accepted.any() else None)
print("Accepted test images:", int(test_accepted.sum()), "/", len(test_df))
print("Test assessment coverage:", test_coverage)
print("Accepted test accuracy:", accepted_test_accuracy)


## CELL 56 — Build detailed test-prediction table

**Repair record:** Completed the truncated result columns used by error analysis and final CSV export.


In [ ]:
test_results = test_df.reset_index(drop=True).copy()
test_results["predicted_class_id"] = test_predictions
test_results["predicted_class"] = [CLASS_NAMES[class_id] for class_id in test_predictions]
test_results["rough_bark_score"] = test_probabilities[:, 0]
test_results["stripe_canker_score"] = test_probabilities[:, 1]
test_results["confidence"] = test_confidence
test_results["margin"] = test_margin
test_results["correct"] = test_predictions == y_test
test_results["accepted"] = test_accepted
test_results["status"] = np.where(test_accepted, "disease_detected", "uncertain")
display(test_results.head())


## CELL 57 — Show misclassifications

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
mistakes = test_results[~test_results["correct"]].copy()
print("Misclassified test images:", len(mistakes))
display(mistakes[["filename", "class_name", "predicted_class", "confidence", "margin"]])


## CELL 58 — Display misclassified images

**Repair record:** Completed the truncated error-image display and empty-mistakes handling.


In [ ]:
NUMBER_TO_SHOW = min(12, len(mistakes))
if NUMBER_TO_SHOW > 0:
    number_of_rows = math.ceil(NUMBER_TO_SHOW / 4)
    fig, axes = plt.subplots(number_of_rows, 4, figsize=(16, 4 * number_of_rows), squeeze=False)
    axes = axes.reshape(-1)
    for index, (_, row) in enumerate(mistakes.head(NUMBER_TO_SHOW).iterrows()):
        with Image.open(row["path"]) as image:
            axes[index].imshow(ImageOps.exif_transpose(image).convert("RGB"))
        axes[index].set_title(f"True: {row['class_name']}\nPredicted: {row['predicted_class']}")
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No misclassified images in this test partition.")


## CELL 59 — Single image preprocessing helper

**Repair record:** Completed the truncated return tuple while retaining the same preprocessing as training.


In [ ]:
def preprocess_single_image(image_path):
    with Image.open(image_path) as image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        original_image = image.copy()
        image = image.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
        array = np.asarray(image, dtype=np.float32)
    batch = np.expand_dims(array, axis=0)
    return original_image, batch


## CELL 60 — Application-style prediction function

**Repair record:** Completed margin, policy decision and application-style response. The function never returns healthy and does not claim out-of-domain rejection.


In [ ]:
def predict_bark_disease(image_path, trained_model=best_model):
    _, batch = preprocess_single_image(image_path)
    probabilities = np.asarray(trained_model.predict(batch, verbose=0)[0], dtype=np.float32)
    if probabilities.shape != (2,) or not np.isfinite(probabilities).all():
        raise ValueError("Invalid two-class probability output.")
    ranking = np.argsort(probabilities)[::-1]
    best_index, second_index = int(ranking[0]), int(ranking[1])
    confidence = float(probabilities[best_index])
    margin = float(probabilities[best_index] - probabilities[second_index])
    accepted = bool(ACCEPTANCE_ENABLED and confidence >= CONFIDENCE_THRESHOLD and margin >= MARGIN_THRESHOLD)
    return {
        "status": "disease_detected" if accepted else "uncertain",
        "prediction": CLASS_NAMES[best_index] if accepted else None,
        "candidate_class": CLASS_NAMES[best_index],
        "confidence_score": confidence, "confidence_margin": margin,
        "review_recommended": not accepted,
        "top_predictions": [{"class": CLASS_NAMES[int(index)], "confidence": float(probabilities[index])}
                            for index in ranking],
        "healthy_detection_supported": False,
        "assessment_scope": "classification_between_two_dataset_disease_labels",
    }


## CELL 61 — Test inference function

This function deliberately has no healthy result.

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
example_image = test_df.iloc[0]["path"]
result = predict_bark_disease(example_image)
print(json.dumps(result, indent=2))


## CELL 62 — Grad-CAM function

**Repair record:** Completed the truncated Grad-CAM graph and gradient calculation through the nested backbone and existing classifier head.


In [ ]:
def make_gradcam_heatmap(image_batch, trained_model, class_index=None):
    backbone_model = trained_model.get_layer(BACKBONE_NAME)
    grad_model = tf.keras.Model(backbone_model.input, [
        backbone_model.get_layer("top_conv").output, backbone_model.output,
    ])
    pooling_layer = trained_model.get_layer("global_average_pooling")
    dropout_layer = trained_model.get_layer("classification_dropout")
    prediction_layer = trained_model.get_layer("bark_predictions")
    image_tensor = tf.convert_to_tensor(image_batch)
    with tf.GradientTape() as tape:
        tape.watch(image_tensor)
        conv_outputs, features = grad_model(image_tensor, training=False)
        pooled = pooling_layer(features)
        probabilities = prediction_layer(dropout_layer(pooled, training=False))
        target_index = (int(tf.argmax(probabilities[0])) if class_index is None else int(class_index))
        class_score = probabilities[:, target_index]
    gradients = tape.gradient(class_score, conv_outputs)
    if gradients is None:
        raise RuntimeError("Grad-CAM gradients are unavailable for this graph.")
    pooled_gradients = tf.reduce_mean(gradients, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(conv_outputs[0] * pooled_gradients, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = tf.math.divide_no_nan(heatmap, tf.reduce_max(heatmap))
    return heatmap.numpy(), probabilities[0].numpy()


## CELL 63 — Display a Grad-CAM example

Inspect model attention visually; Grad-CAM is not independent diagnostic validation.

**Repair record:** Completed the missing heatmap resize/overlay; retained the model softmax output and explained the visualization limit.


In [ ]:
example_image = test_df.iloc[0]["path"]
original_image, batch = preprocess_single_image(example_image)
heatmap, probabilities = make_gradcam_heatmap(batch, best_model)
prediction_index = int(np.argmax(probabilities))
print("Prediction:", CLASS_NAMES[prediction_index])
print("Probabilities:", probabilities)
heatmap_image = Image.fromarray(np.uint8(np.clip(heatmap, 0, 1) * 255))
heatmap_image = heatmap_image.resize(original_image.size, Image.Resampling.BILINEAR)
plt.figure(figsize=(7, 7))
plt.imshow(original_image)
plt.imshow(np.asarray(heatmap_image), cmap="jet", alpha=0.40, vmin=0, vmax=255)
plt.title(f"Grad-CAM: {CLASS_NAMES[prediction_index]}")
plt.axis("off")
plt.show()
print("A saliency map is an inspection aid, not independent proof of a diagnosis.")


## CELL 64 — Create final handoff directories

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
HANDOFF_DIR = Path("/content/bark_model_handoff")
if HANDOFF_DIR.exists():
    shutil.rmtree(HANDOFF_DIR)
EVALUATION_DIR = HANDOFF_DIR / "evaluation"
PARITY_DIR = HANDOFF_DIR / "parity_fixtures"
PARITY_IMAGES_DIR = PARITY_DIR / "images"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
PARITY_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
print(HANDOFF_DIR)


## CELL 65 — Save final Keras model

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
KERAS_MODEL_PATH = HANDOFF_DIR / "bark_condition_model.keras"
best_model.save(KERAS_MODEL_PATH)
print("Saved:", KERAS_MODEL_PATH)


## CELL 66 — Save companion HDF5 model

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
H5_MODEL_PATH = HANDOFF_DIR / "bark_condition_model.h5"
best_model.save(H5_MODEL_PATH, include_optimizer=False)
print("Saved:", H5_MODEL_PATH)


## CELL 67 — Save exact class order

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
with open(HANDOFF_DIR / "class_names.json", "w", encoding="utf-8") as file:
    json.dump(CLASS_NAMES, file, indent=2)
print(CLASS_NAMES)


## CELL 68 — Save label definitions

**Repair record:** Completed the truncated label-definitions JSON writer; preserved the exact two labels.


In [ ]:
label_definitions = {
    "RoughBark": {"display_name": "Rough Bark", "type": "supported_disease_label",
                  "dataset_label": "RoughBark", "healthy": False},
    "StripeCanker": {"display_name": "Stripe Canker", "type": "supported_disease_label",
                     "dataset_label": "StripeCanker", "healthy": False},
}
with open(HANDOFF_DIR / "label_definitions.json", "w", encoding="utf-8") as file:
    json.dump(label_definitions, file, indent=2)


## CELL 69 — Save backend preprocessing.py

**Repair record:** Restored line breaks inside the generated Python string and added the missing file write.


In [ ]:
preprocessing_code = '''from PIL import Image, ImageOps
import numpy as np

IMAGE_SIZE = 224

def preprocess_bark_image(path_or_file):
    with Image.open(path_or_file) as image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
        array = np.asarray(image, dtype=np.float32)
    return np.expand_dims(array, axis=0)
'''
(HANDOFF_DIR / "preprocessing.py").write_text(preprocessing_code, encoding="utf-8")


## CELL 70 — Save training environment

**Repair record:** Completed environment data and JSON writer; recorded actual package/platform/build versions instead of installing target-backend pins.


In [ ]:
training_environment = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__,
    "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
    "pillow": PIL.__version__, "h5py": h5py.__version__,
    "kagglehub": package_metadata.version("kagglehub"),
    "matplotlib": package_metadata.version("matplotlib"),
    "platform": platform.platform(),
    "gpu_devices": [str(device) for device in tf.config.list_physical_devices("GPU")],
    "tensorflow_build": {key: str(value) for key, value in tf.sysconfig.get_build_info().items()},
}
with open(HANDOFF_DIR / "training_environment.json", "w", encoding="utf-8") as file:
    json.dump(training_environment, file, indent=2)
display(training_environment)


## CELL 71 — Save dataset summary

**Repair record:** Completed dataset-summary fields and writer; records excluded cross-label conflicts and exports their audit alongside near-duplicate/invalid-image audits.


In [ ]:
path_to_split = dict(zip(split_manifest["path"], split_manifest["split"]))
cross_split_near_pairs = int(sum(path_to_split.get(row.path_a) != path_to_split.get(row.path_b)
                                for row in near_duplicate_df.itertuples()))
dataset_summary = {
    "source": "Kaggle", "kaggle_dataset_id": KAGGLE_DATASET_ID,
    "dataset_directory": str(DATASET_DIR), "download_cache_path": str(DATASET_ROOT),
    "total_usable_images": int(len(df)), "invalid_images_removed": int(len(bad_images)),
    "exact_duplicates_removed": int(DUPLICATES_REMOVED),
    "conflicting_label_hash_groups_excluded": int(CONFLICTING_HASH_GROUPS_REMOVED),
    "conflicting_label_images_excluded": int(CONFLICTING_IMAGES_REMOVED),
    "conflict_resolution_policy": "Exclude every copy of a cross-label identical file; do not infer its correct label.",
    "potential_near_duplicate_pairs": int(len(near_duplicate_df)),
    "near_duplicate_audit_complete": bool(near_duplicate_audit_complete),
    "near_duplicate_cross_split_pairs_reported": cross_split_near_pairs,
    "class_counts": {name: int((df["class_name"] == name).sum()) for name in CLASS_NAMES},
    "partitions": {name: {"images": int(len(part)), "class_counts": {
        label: int((part["class_name"] == label).sum()) for label in CLASS_NAMES}}
        for name, part in [("train", train_df), ("validation", validation_df), ("test", test_df)]},
    "split_method": "stratified_image_level_after_exact_file_deduplication",
    "split_seed": SEED, "healthy_detection_supported": False,
    "tree_site_independence_established": False,
    "limitations": ["No healthy class is supplied.", "Tree/site IDs are not provided by this workflow.",
                    "File hashes do not detect all re-encoded duplicates.",
                    "Near-duplicate pairs are reported only; related-image leakage may remain.",
                    "Dataset disease labels have not been independently expert-confirmed by this notebook."],
}
with open(HANDOFF_DIR / "dataset_summary.json", "w", encoding="utf-8") as file:
    json.dump(dataset_summary, file, indent=2)
near_duplicate_df.to_csv(EVALUATION_DIR / "near_duplicate_audit.csv", index=False)
bad_images.to_csv(EVALUATION_DIR / "invalid_images.csv", index=False)
conflicting_images.to_csv(EVALUATION_DIR / "conflicting_label_images.csv", index=False)


## CELL 72 — Copy training history and split manifest

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
split_manifest.to_csv(HANDOFF_DIR / "split_manifest.csv", index=False)
training_history.to_csv(HANDOFF_DIR / "training_history.csv", index=False)


## CELL 73 — Save classification report

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
classification_report_dict = classification_report(
    y_test, test_predictions, labels=[0, 1], target_names=CLASS_NAMES, output_dict=True, zero_division=0,
)
with open(EVALUATION_DIR / "classification_report.json", "w", encoding="utf-8") as file:
    json.dump(classification_report_dict, file, indent=2)


## CELL 74 — Save test predictions

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
test_results.to_csv(EVALUATION_DIR / "test_predictions.csv", index=False)


## CELL 75 — Save confusion matrix CSV and PNG

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
confusion_df = pd.DataFrame(confusion, index=CLASS_NAMES, columns=CLASS_NAMES)
confusion_df.to_csv(EVALUATION_DIR / "confusion_matrix.csv")
fig, ax = plt.subplots(figsize=(7, 7))
display_object = ConfusionMatrixDisplay(confusion_matrix=confusion, display_labels=CLASS_NAMES)
display_object.plot(ax=ax, values_format="d")
plt.title("Cinnamon Bark Disease Confusion Matrix")
plt.savefig(EVALUATION_DIR / "confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.close(fig)


## CELL 76 — Save threshold calibration

**Repair record:** Completed the policy export and documented validation targets, fallback and unsupported healthy/OOD claims.


In [ ]:
threshold_policy = {
    "confidence_threshold": CONFIDENCE_THRESHOLD, "margin_threshold": MARGIN_THRESHOLD,
    "selected_using": "validation_data_only", "target_accepted_accuracy": TARGET_ACCEPTED_ACCURACY,
    "minimum_validation_coverage": MIN_VALIDATION_COVERAGE,
    "acceptance_enabled": bool(ACCEPTANCE_ENABLED), "validation_targets_met": bool(ACCEPTANCE_ENABLED),
    "selected_validation_coverage": float(selected_threshold["coverage"]),
    "selected_validation_accepted_accuracy": float(selected_threshold["accepted_accuracy"]),
    "accepted_result": "disease_detected", "unaccepted_result": "uncertain",
    "healthy_detection_supported": False,
    "note": "These thresholds only accept RoughBark/StripeCanker classification. They do not distinguish healthy bark or reject all unknown inputs.",
    "fallback": "Disable acceptance when no validation policy meets the stated targets.",
}
with open(EVALUATION_DIR / "threshold_calibration.json", "w", encoding="utf-8") as file:
    json.dump(threshold_policy, file, indent=2)
threshold_df.to_csv(EVALUATION_DIR / "validation_threshold_search.csv", index=False)


## CELL 77 — Save final metrics

**Repair record:** Completed truncated metrics with actual test support/coverage; unsupported health accuracy remains null.


In [ ]:
metrics = {
    "chosen_training_stage": chosen_stage,
    "stage1_validation_accuracy": float(stage1_accuracy), "stage1_validation_macro_f1": float(stage1_macro_f1),
    "stage2_validation_accuracy": float(stage2_accuracy), "stage2_validation_macro_f1": float(stage2_macro_f1),
    "test_accuracy": float(test_accuracy), "test_macro_f1": float(test_macro_f1),
    "test_balanced_accuracy": float(test_balanced_accuracy), "test_images": int(len(test_df)),
    "test_accepted_images": int(test_accepted.sum()), "test_uncertain_images": int((~test_accepted).sum()),
    "test_coverage": test_coverage, "accepted_test_accuracy": accepted_test_accuracy,
    "test_class_support": {name: int((y_test == i).sum()) for i, name in enumerate(CLASS_NAMES)},
    "healthy_detection_supported": False, "healthy_unhealthy_accuracy": None,
    "model_selection_source": "validation_macro_f1", "threshold_source": "validation_only",
}
with open(EVALUATION_DIR / "metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2, allow_nan=False)


## CELL 78 — Save error analysis

**Repair record:** Completed the truncated error-analysis writer and documented the retained split/domain limitations.


In [ ]:
error_lines = ["# Bark Model Error Analysis", "", f"Selected model: {chosen_stage}", "",
               f"Test images: {len(test_df)}", f"Misclassified images: {len(mistakes)}", "",
               "## Misclassified images", ""]
if mistakes.empty:
    error_lines.append("No misclassified images were observed in this test partition.")
else:
    for _, row in mistakes.iterrows():
        error_lines.append(f"- {row['filename']}: true={row['class_name']}, predicted={row['predicted_class']}, "
                           f"confidence={row['confidence']:.4f}, margin={row['margin']:.4f}")
error_lines += ["", "## Scope and limitations", "",
                f"Cross-label identical images excluded before splitting: {CONFLICTING_IMAGES_REMOVED} ({CONFLICTING_HASH_GROUPS_REMOVED} hash groups).",
                "Healthy bark is not supported; unknown uploads may still receive high scores.",
                "Image-level splitting does not establish unseen-tree/site generalization.",
                f"Potential near-duplicate cross-split pairs reported: {cross_split_near_pairs}.",
                "Do not change the model/thresholds after examining these test errors and reuse this test as independent."]
(EVALUATION_DIR / "error_analysis.md").write_text("\n".join(error_lines) + "\n", encoding="utf-8")


## CELL 79 — Create real-image parity fixture set

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
fixture_parts = [test_df[test_df["class_name"] == class_name].head(4) for class_name in CLASS_NAMES]
fixture_df = pd.concat(fixture_parts, ignore_index=True)
assert set(fixture_df["class_name"]) == set(CLASS_NAMES)
display(fixture_df[["filename", "class_name"]])


## CELL 80 — Copy parity images

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
fixture_file_names = []
for index, row in fixture_df.iterrows():
    source_path = Path(row["path"])
    output_name = f"{index:02d}_{row['class_name']}_{source_path.name}"
    shutil.copy2(source_path, PARITY_IMAGES_DIR / output_name)
    fixture_file_names.append(output_name)
print("Parity images copied:", len(fixture_file_names))


## CELL 81 — Calculate original fixture predictions

**Repair record:** Completed fixture records with image/tensor hashes, repeated-inference checks and expected uncertainty decisions.


In [ ]:
fixture_expected = []
for index, row in fixture_df.iterrows():
    _, batch = preprocess_single_image(row["path"])
    probabilities = np.asarray(best_model.predict(batch, verbose=0)[0], dtype=np.float32)
    repeated = np.asarray(best_model.predict(batch, verbose=0)[0], dtype=np.float32)
    np.testing.assert_allclose(probabilities, repeated, atol=1e-5, rtol=0)
    ordered = np.sort(probabilities)
    fixture_expected.append({
        "file": fixture_file_names[index], "true_class": row["class_name"],
        "image_sha256": sha256_file(PARITY_IMAGES_DIR / fixture_file_names[index]),
        "input_tensor_sha256": hashlib.sha256(batch.astype("<f4").tobytes(order="C")).hexdigest(),
        "expected_probabilities": probabilities.tolist(),
        "predicted_class": CLASS_NAMES[int(np.argmax(probabilities))],
        "expected_accepted": bool(ACCEPTANCE_ENABLED and ordered[-1] >= CONFIDENCE_THRESHOLD
                                   and ordered[-1] - ordered[-2] >= MARGIN_THRESHOLD),
    })


## CELL 82 — Reload .keras and compare predictions

**Repair record:** Completed the truncated difference loop and missing aggregate variables; also checks uncertainty-policy agreement.


In [ ]:
keras_reloaded = tf.keras.models.load_model(KERAS_MODEL_PATH, compile=False)
keras_differences, keras_label_matches, keras_policy_matches = [], [], []
for fixture, (_, row) in zip(fixture_expected, fixture_df.iterrows()):
    _, batch = preprocess_single_image(row["path"])
    original_prediction = np.asarray(fixture["expected_probabilities"], dtype=np.float32)
    reloaded_prediction = np.asarray(keras_reloaded.predict(batch, verbose=0)[0], dtype=np.float32)
    keras_differences.append(float(np.max(np.abs(original_prediction - reloaded_prediction))))
    keras_label_matches.append(bool(np.argmax(original_prediction) == np.argmax(reloaded_prediction)))
    scores = np.sort(reloaded_prediction)
    accepted = bool(ACCEPTANCE_ENABLED and scores[-1] >= CONFIDENCE_THRESHOLD
                    and scores[-1] - scores[-2] >= MARGIN_THRESHOLD)
    keras_policy_matches.append(accepted == fixture["expected_accepted"])
KERAS_MAX_DIFFERENCE = float(max(keras_differences))
KERAS_LABEL_AGREEMENT = float(np.mean(keras_label_matches))
KERAS_POLICY_AGREEMENT = float(np.mean(keras_policy_matches))
print(".keras maximum difference:", KERAS_MAX_DIFFERENCE)


## CELL 83 — Reload .h5 and compare predictions

**Repair record:** Completed the truncated HDF5 parity loop and missing aggregate variables; also checks uncertainty-policy agreement.


In [ ]:
h5_reloaded = tf.keras.models.load_model(H5_MODEL_PATH, compile=False)
h5_differences, h5_label_matches, h5_policy_matches = [], [], []
for fixture, (_, row) in zip(fixture_expected, fixture_df.iterrows()):
    _, batch = preprocess_single_image(row["path"])
    original_prediction = np.asarray(fixture["expected_probabilities"], dtype=np.float32)
    reloaded_prediction = np.asarray(h5_reloaded.predict(batch, verbose=0)[0], dtype=np.float32)
    h5_differences.append(float(np.max(np.abs(original_prediction - reloaded_prediction))))
    h5_label_matches.append(bool(np.argmax(original_prediction) == np.argmax(reloaded_prediction)))
    scores = np.sort(reloaded_prediction)
    accepted = bool(ACCEPTANCE_ENABLED and scores[-1] >= CONFIDENCE_THRESHOLD
                    and scores[-1] - scores[-2] >= MARGIN_THRESHOLD)
    h5_policy_matches.append(accepted == fixture["expected_accepted"])
H5_MAX_DIFFERENCE = float(max(h5_differences))
H5_LABEL_AGREEMENT = float(np.mean(h5_label_matches))
H5_POLICY_AGREEMENT = float(np.mean(h5_policy_matches))
print(".h5 maximum difference:", H5_MAX_DIFFERENCE)


## CELL 84 — Evaluate export parity

Do not increase tolerance merely to force a pass. Failed formats are retained for diagnosis, not deployment.

**Repair record:** Completed the truncated parity status display and policy checks; reports failures honestly while allowing the diagnostic ZIP to be created.


In [ ]:
PARITY_TOLERANCE = 1e-5
keras_parity_pass = bool(KERAS_MAX_DIFFERENCE <= PARITY_TOLERANCE
                         and KERAS_LABEL_AGREEMENT == 1.0 and KERAS_POLICY_AGREEMENT == 1.0)
h5_parity_pass = bool(H5_MAX_DIFFERENCE <= PARITY_TOLERANCE
                      and H5_LABEL_AGREEMENT == 1.0 and H5_POLICY_AGREEMENT == 1.0)
print(".keras parity:", "PASS" if keras_parity_pass else "REVIEW")
print(".h5 parity:", "PASS" if h5_parity_pass else "REVIEW")
print("Failed artifacts remain in the diagnostic package, but are not approved for deployment.")
print("Do not increase the tolerance just to make an export pass.")


## CELL 85 — Save parity information

**Repair record:** Completed the parity JSON writer; records actual format results and explicitly leaves backend parity untested.


In [ ]:
parity_information = {
    "same_environment_tolerance": PARITY_TOLERANCE,
    "class_names": CLASS_NAMES, "decision_policy": threshold_policy,
    "keras": {"maximum_absolute_difference": KERAS_MAX_DIFFERENCE,
              "label_agreement": KERAS_LABEL_AGREEMENT, "policy_agreement": KERAS_POLICY_AGREEMENT,
              "passed": keras_parity_pass},
    "h5": {"maximum_absolute_difference": H5_MAX_DIFFERENCE,
           "label_agreement": H5_LABEL_AGREEMENT, "policy_agreement": H5_POLICY_AGREEMENT,
           "passed": h5_parity_pass},
    "backend_parity_tested": False, "fixtures": fixture_expected,
}
with open(PARITY_DIR / "expected_predictions.json", "w", encoding="utf-8") as file:
    json.dump(parity_information, file, indent=2, allow_nan=False)


## CELL 86 — Save target CinnaAI backend requirements

These are the previously inspected backend target versions. This cell does not install them in Colab.

**Repair record:** Restored actual newlines between requirements. Target-backend versions are documentation only, not a Colab installation command.


In [ ]:
target_backend_requirements = "\n".join([
    "tensorflow==2.16.2", "keras==3.15.1", "numpy==1.26.4", "h5py==3.16.0", "Pillow==12.2.0",
]) + "\n"
(HANDOFF_DIR / "requirements-inference.txt").write_text(target_backend_requirements, encoding="utf-8")
print("Documented target backend versions; no packages were installed or downgraded by this cell.")


## CELL 87 — Save actual Colab training requirements

**Repair record:** Restored requirement-file newlines and records actual KaggleHub/Matplotlib versions used by training.


In [ ]:
training_requirements = "\n".join([
    f"tensorflow=={tf.__version__}", f"keras=={keras.__version__}", f"numpy=={np.__version__}",
    f"pandas=={pd.__version__}", f"scikit-learn=={sklearn.__version__}", f"Pillow=={PIL.__version__}",
    f"h5py=={h5py.__version__}", f"kagglehub=={training_environment['kagglehub']}",
    f"matplotlib=={training_environment['matplotlib']}",
]) + "\n"
(HANDOFF_DIR / "requirements-training.txt").write_text(training_requirements, encoding="utf-8")


## CELL 88 — Calculate important SHA-256 checksums

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
def sha256_path(file_path):
    hasher = hashlib.sha256()
    with open(file_path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            hasher.update(block)
    return hasher.hexdigest()

KERAS_SHA256 = sha256_path(KERAS_MODEL_PATH)
H5_SHA256 = sha256_path(H5_MODEL_PATH)
SPLIT_SHA256 = sha256_path(HANDOFF_DIR / "split_manifest.csv")


## CELL 89 — Create metadata.json

**Repair record:** Completed the truncated metadata with preprocessing/output/versions/policy/hashes. No backend or deployment success is asserted before testing.


In [ ]:
metadata = {
    "schema_version": 1, "model_version": "bark-disease-v1",
    "task": "supported_bark_disease_classification", "architecture": "EfficientNetB0",
    "model_file": "bark_condition_model.keras", "artifact_sha256": KERAS_SHA256,
    "companion_h5_sha256": H5_SHA256,
    "dataset": {"source": "Kaggle", "dataset_id": KAGGLE_DATASET_ID},
    "input": {"shape": [None, 224, 224, 3], "dtype": "float32", "colour_order": "RGB",
              "pixel_range": [0, 255], "exif_transpose": True, "resize_library": "Pillow",
              "resize_method": "bilinear", "aspect_ratio_policy": "stretch",
              "external_normalization": "none", "backbone_preprocessing": "included_in_model"},
    "output": {"shape": [None, 2], "activation": "softmax", "class_names": CLASS_NAMES},
    "decision_policy": threshold_policy,
    "training": {"seed": SEED, "chosen_stage": chosen_stage, "split_unit": "image",
                 "split_manifest_sha256": SPLIT_SHA256, "fine_tune_fraction": FINE_TUNE_FRACTION,
                 "checkpoint_selection": "validation_loss", "stage_selection": "validation_macro_f1"},
    "training_versions": training_environment,
    "target_backend_versions": {"tensorflow": "2.16.2", "keras": "3.15.1", "numpy": "1.26.4",
                                "h5py": "3.16.0", "pillow": "12.2.0"},
    "tested_backend_versions": None, "backend_parity_tested": False,
    "export_parity": {"keras_passed": keras_parity_pass, "h5_passed": h5_parity_pass},
    "deployment_ready": False,
    "deployment_note": "Backend parity and target-domain validation remain required; failed exports are diagnostic only.",
    "healthy_detection_supported": False, "evaluation": metrics,
    "limitations": dataset_summary["limitations"] + [
        "Confidence does not establish healthy bark or reliable rejection of unrelated images.",
        "No production backend inference has been tested by this Colab run.",
    ],
}
with open(HANDOFF_DIR / "metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, allow_nan=False)


## CELL 90 — Create backend parity test script

**Repair record:** Completed the truncated generated parity script and writer. Supports either export and verifies model/image/tensor hashes, scores, labels and policy decisions.


In [ ]:
backend_parity_script = r'''import argparse
import hashlib
import json
from pathlib import Path
import numpy as np
import tensorflow as tf
from preprocessing import preprocess_bark_image

BASE_DIR = Path(__file__).resolve().parent
parser = argparse.ArgumentParser(description="Verify bark export predictions on real-image fixtures.")
parser.add_argument("--model", default="bark_condition_model.keras")
args = parser.parse_args()
MODEL_PATH = BASE_DIR / args.model
with open(BASE_DIR / "parity_fixtures" / "expected_predictions.json", encoding="utf-8") as file:
    parity_data = json.load(file)
with open(BASE_DIR / "metadata.json", encoding="utf-8") as file:
    metadata = json.load(file)
with open(BASE_DIR / "class_names.json", encoding="utf-8") as file:
    class_names = json.load(file)
assert class_names == ["RoughBark", "StripeCanker"] == parity_data["class_names"]
TOLERANCE = 1e-4
model_hash = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
expected_hash = (metadata["companion_h5_sha256"] if MODEL_PATH.suffix == ".h5" else metadata["artifact_sha256"])
assert model_hash == expected_hash, "Model file checksum differs from metadata."
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
assert model.input_shape == (None, 224, 224, 3)
assert model.output_shape == (None, 2)
policy = parity_data["decision_policy"]
differences = []
for fixture in parity_data["fixtures"]:
    image_path = BASE_DIR / "parity_fixtures" / "images" / fixture["file"]
    assert hashlib.sha256(image_path.read_bytes()).hexdigest() == fixture["image_sha256"]
    batch = preprocess_bark_image(image_path)
    tensor_hash = hashlib.sha256(batch.astype("<f4").tobytes(order="C")).hexdigest()
    assert tensor_hash == fixture["input_tensor_sha256"], "Preprocessing pixels differ."
    actual = np.asarray(model(batch, training=False))[0]
    expected = np.asarray(fixture["expected_probabilities"], dtype=np.float32)
    assert np.isfinite(actual).all() and ((actual >= 0) & (actual <= 1)).all()
    np.testing.assert_allclose(actual.sum(), 1.0, atol=1e-5, rtol=0)
    np.testing.assert_allclose(actual, expected, atol=TOLERANCE, rtol=0)
    np.testing.assert_allclose(actual, np.asarray(model(batch, training=False))[0], atol=TOLERANCE, rtol=0)
    assert class_names[int(np.argmax(actual))] == fixture["predicted_class"]
    scores = np.sort(actual)
    accepted = bool(policy["acceptance_enabled"] and scores[-1] >= policy["confidence_threshold"]
                    and scores[-1] - scores[-2] >= policy["margin_threshold"])
    assert accepted == fixture["expected_accepted"], "Uncertainty decision changed."
    differences.append(float(np.max(np.abs(actual - expected))))
print("PASS:", MODEL_PATH.name, "fixtures:", len(differences), "max score difference:", max(differences))
print("This verifies numerical compatibility only, not field disease accuracy.")
'''
(HANDOFF_DIR / "backend_parity_check.py").write_text(backend_parity_script, encoding="utf-8")


## CELL 91 — Create README for Codex

**Repair record:** Completed the truncated handoff README, including results, exact input/output, failed-export handling and integration instructions.


In [ ]:
readme_text = f"""# CinnaAI Bark Disease Model

## Supported disease classes
1. RoughBark
2. StripeCanker

Exact output order: {CLASS_NAMES}
No healthy bark class is present. A high score is not evidence that an unrelated image is diseased.

## Selected model
{chosen_stage}; selected using validation macro-F1, with validation-loss checkpoints within each stage.

## Test results
Accuracy: {test_accuracy:.4f}
Macro F1: {test_macro_f1:.4f}
Balanced accuracy: {test_balanced_accuracy:.4f}
Accepted coverage: {test_coverage:.4f}
Accepted accuracy: {accepted_test_accuracy}

## Input contract
- RGB, 224 x 224, float32, values 0-255
- EXIF orientation correction; Pillow bilinear stretched resize
- No external division by 255; EfficientNetB0 preprocessing is inside the model
- Random augmentation is training-only and is absent from exported models

## Output contract
Two-column softmax in the exact class order above.
Accepted result: disease_detected. Other result: uncertain.
Acceptance enabled: {ACCEPTANCE_ENABLED}; confidence >= {CONFIDENCE_THRESHOLD}, margin >= {MARGIN_THRESHOLD}.
When validation targets are unmet, all application results remain uncertain.

## Export checks
Native Keras parity passed: {keras_parity_pass}; maximum difference: {KERAS_MAX_DIFFERENCE}.
HDF5 parity passed: {h5_parity_pass}; maximum difference: {H5_MAX_DIFFERENCE}.
Files that fail parity are diagnostic exports and must not be deployed.
Backend compatibility is UNTESTED. Training and target-backend requirements are separate.

## Backend verification
Use the provided preprocessing.py and run:
python backend_parity_check.py
python backend_parity_check.py --model bark_condition_model.h5
The 1e-4 cross-runtime score tolerance does not excuse changed pixels, labels or decisions.

## Dataset and evaluation limits
Dataset: {KAGGLE_DATASET_ID}
Image-level stratified split after exact-file deduplication; no confirmed tree/site-independent test.
Cross-label identical images excluded before splitting: {CONFLICTING_IMAGES_REMOVED} ({CONFLICTING_HASH_GROUPS_REMOVED} hash groups).
Every conflicting copy is excluded; original dataset files are unchanged. See evaluation/conflicting_label_images.csv.
Potential cross-split near-duplicate pairs reported: {cross_split_near_pairs}.
The hash heuristic reports suspicious pairs; it does not remove or group them.
Healthy/unhealthy accuracy and healthy rejection have not been evaluated.
Do not tune the model or policy using this test partition and then claim it remains independent.

## Integration
Add a separate authenticated bark endpoint and history. Do not replace the seven-class leaf model.
Inspect metadata.json, metrics, threshold_calibration.json and parity fixtures before integration.
checksums.sha256 covers every package file except the checksum file itself.
"""
(HANDOFF_DIR / "README.md").write_text(readme_text, encoding="utf-8")


## CELL 92 — Create complete file checksums

All output files must exist before checksums. Do not modify files after this cell without rerunning checksums and ZIP creation.

**Repair record:** Completed the truncated checksum writer, with real newlines and portable relative paths.


In [ ]:
checksum_lines = []
for file_path in sorted(HANDOFF_DIR.rglob("*")):
    if file_path.is_file() and file_path.name != "checksums.sha256":
        relative_path = file_path.relative_to(HANDOFF_DIR).as_posix()
        checksum_lines.append(f"{sha256_path(file_path)}  {relative_path}")
(HANDOFF_DIR / "checksums.sha256").write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")
print("Checksummed files:", len(checksum_lines))


## CELL 93 — Display final handoff package

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
print("FINAL MODEL HANDOFF CONTENTS")
print("=" * 70)
for file_path in sorted(HANDOFF_DIR.rglob("*")):
    if file_path.is_file():
        print(file_path.relative_to(HANDOFF_DIR))


## CELL 94 — Create final ZIP

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
FINAL_ZIP_PATH = shutil.make_archive("/content/bark_model_handoff", "zip", root_dir=HANDOFF_DIR)
print("Final ZIP created:", FINAL_ZIP_PATH)
print("Save the ZIP before disconnecting or resetting this Colab runtime.")


## CELL 95 — Download final model package

Download and preserve `bark_model_handoff.zip`.

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
from google.colab import files
files.download(FINAL_ZIP_PATH)


## CELL 96 — Optional manual prediction test

Optional: enable the flag after training and run CELL 96 followed by CELL 97. This is not a new independent evaluation.

**Repair record:** Guarded the optional interactive upload so top-to-bottom Run all does not block on an unnecessary manual test.


In [ ]:
from google.colab import files
# Disabled by default so Run all completes without waiting for an optional upload.
RUN_OPTIONAL_MANUAL_TEST = False
uploaded = files.upload() if RUN_OPTIONAL_MANUAL_TEST else {}
if not RUN_OPTIONAL_MANUAL_TEST:
    print("Optional test skipped. Set RUN_OPTIONAL_MANUAL_TEST=True and run CELL 96 then CELL 97 to upload an image.")


## CELL 97 — Predict uploaded image

**Repair record:** Added the empty-upload guard corresponding to the explicitly optional manual test.


In [ ]:
if uploaded:
    uploaded_filename = next(iter(uploaded))
    prediction_result = predict_bark_disease(uploaded_filename)
    print(json.dumps(prediction_result, indent=2))
else:
    print("No optional image was uploaded. Training and handoff creation are complete.")


## Reconstruction log

- The exact Kaggle ID and class order remain unchanged; no healthy label is introduced.
- Missing bodies and file writers are reconstructed; strings contain actual newlines.
- Stage 2 reloads an independent Stage 1 checkpoint, preventing fine-tuning from destroying the Stage 1 candidate.
- Inference has no random augmentation or external `/255`; training uses the supplied mild transformations.
- Near-duplicate auditing reports heuristic pairs with resource caps; it does not claim tree independence or silently delete data.
- Dataset caching uses disk to limit RAM pressure.
- Unmet validation policy targets disable automatic acceptance; uncertainty is preserved rather than inventing a healthy result.
- Export parity checks compare scores, labels and uncertainty decisions. Failed formats remain diagnostic in the ZIP.
- Colab versions are recorded; old backend target pins are written as documentation only.
- The optional upload is disabled by default for uninterrupted Run all.

## Per-cell record

| CELL | Supplied title | Repair |
|---:|---|---|
| 1 | Check GPU | Preserved the supplied Colab shell command. |
| 2 | Install only missing packages | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 3 | Imports | Restored imports; added importlib.metadata to record installed package versions without changing packages. |
| 4 | Print and record Colab environment | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 5 | Reproducibility settings | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 6 | Download dataset from Kaggle | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 7 | Display dataset structure | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 8 | Automatically locate the Diseases folder | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 9 | Define the exact model class order | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 10 | Build image manifest | Completed the class-count display and supplied explicit columns so an empty manifest receives a clear validation error. |
| 11 | Make sure both classes contain enough images | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 12 | Validate every image | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 13 | Remove invalid images | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 14 | Calculate SHA-256 hashes | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 15 | Find exact duplicates | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 16 | Detect duplicate images with conflicting labels | Exclude all copies of cross-label identical files from the in-memory manifest, preserve originals, and retain an audit table instead of stopping or selecting a label arbitrarily. |
| 17 | Remove duplicate copies | Completed duplicate removal and post-cleaning class checks; added a pre-deduplication guard so CELL 17 cannot silently choose a label for an unresolved cross-label file. |
| 18 | Near-duplicate audit | The source only defined a hash function. Added candidate comparison, a results table, and explicit resource-cap/completeness reporting; no automatic deletion. |
| 19 | Plot class distribution | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 20 | Display random images from each class | Completed the truncated image display loop and hid unused axes. |
| 21 | Create working directory | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 22 | Create 70/15/15 train-validation-test split | Completed test partition reporting; retained the supplied image-level split and its independence limitation. |
| 23 | Verify no exact duplicate leakage | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 24 | Save permanent split manifest | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 25 | Define exact bark preprocessing | Completed the missing tf_load_image body with static shape and label dtype; preserved EXIF/RGB/Pillow bilinear/[0,255]. |
| 26 | Define training-only augmentation | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 27 | Build TensorFlow datasets | Completed batching, training-only shuffle/augmentation and all three dataset definitions. Used disk cache and clipped augmented pixels to [0,255]. |
| 28 | Verify model input tensor | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 29 | Display augmented training samples | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 30 | Calculate class weights | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 31 | Build EfficientNetB0 model | Completed the missing pooling/dropout/two-output softmax head and model construction. No augmentation or external rescaling in the inference graph. |
| 32 | Verify model input/output contract | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 33 | Compile Stage 1 | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 34 | Stage 1 callbacks | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 35 | Train Stage 1 | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 36 | Prediction helper | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 37 | Load and evaluate best Stage 1 checkpoint | Completed the truncated validation output, retaining the best validation-loss checkpoint and recording all comparison metrics. |
| 38 | Start Stage 2 from the BEST Stage 1 model | Fixed the supplied alias model=stage1_model: reload an independent checkpoint before fine-tuning so Stage 1 remains selectable. Explicitly set both frozen and unfrozen flags. |
| 39 | Recompile Stage 2 | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 40 | Stage 2 callbacks | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 41 | Train Stage 2 | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 42 | Evaluate Stage 2 | Completed the truncated validation output using the best Stage 2 checkpoint. |
| 43 | Compare both training stages | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 44 | Automatically select the better model | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 45 | Plot Stage 1 accuracy | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 46 | Plot Stage 1 loss | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 47 | Plot Stage 2 accuracy | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 48 | Plot Stage 2 loss | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 49 | Combine and save training history | Closed the truncated training-history export call. |
| 50 | Calculate validation confidence and margin | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 51 | Search validation thresholds | Completed the truncated threshold search, accepted-accuracy/coverage calculations and missing threshold_df definition. |
| 52 | Automatically choose uncertainty threshold | Completed threshold selection and scalar definitions. Documented a conservative missing-target fallback: disable acceptance instead of claiming unmet 90%/60% goals. |
| 53 | FINAL held-out test evaluation | Completed the truncated held-out evaluation and per-class classification report; no healthy-class metrics are invented. |
| 54 | Test confusion matrix | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 55 | Apply uncertainty policy to test results | Completed truncated accepted-test metrics and handled zero accepted examples without NaN or division errors. |
| 56 | Build detailed test-prediction table | Completed the truncated result columns used by error analysis and final CSV export. |
| 57 | Show misclassifications | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 58 | Display misclassified images | Completed the truncated error-image display and empty-mistakes handling. |
| 59 | Single image preprocessing helper | Completed the truncated return tuple while retaining the same preprocessing as training. |
| 60 | Application-style prediction function | Completed margin, policy decision and application-style response. The function never returns healthy and does not claim out-of-domain rejection. |
| 61 | Test inference function | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 62 | Grad-CAM function | Completed the truncated Grad-CAM graph and gradient calculation through the nested backbone and existing classifier head. |
| 63 | Display a Grad-CAM example | Completed the missing heatmap resize/overlay; retained the model softmax output and explained the visualization limit. |
| 64 | Create final handoff directories | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 65 | Save final Keras model | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 66 | Save companion HDF5 model | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 67 | Save exact class order | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 68 | Save label definitions | Completed the truncated label-definitions JSON writer; preserved the exact two labels. |
| 69 | Save backend preprocessing.py | Restored line breaks inside the generated Python string and added the missing file write. |
| 70 | Save training environment | Completed environment data and JSON writer; recorded actual package/platform/build versions instead of installing target-backend pins. |
| 71 | Save dataset summary | Completed dataset-summary fields and writer; records excluded cross-label conflicts and exports their audit alongside near-duplicate/invalid-image audits. |
| 72 | Copy training history and split manifest | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 73 | Save classification report | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 74 | Save test predictions | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 75 | Save confusion matrix CSV and PNG | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 76 | Save threshold calibration | Completed the policy export and documented validation targets, fallback and unsupported healthy/OOD claims. |
| 77 | Save final metrics | Completed truncated metrics with actual test support/coverage; unsupported health accuracy remains null. |
| 78 | Save error analysis | Completed the truncated error-analysis writer and documented the retained split/domain limitations. |
| 79 | Create real-image parity fixture set | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 80 | Copy parity images | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 81 | Calculate original fixture predictions | Completed fixture records with image/tensor hashes, repeated-inference checks and expected uncertainty decisions. |
| 82 | Reload .keras and compare predictions | Completed the truncated difference loop and missing aggregate variables; also checks uncertainty-policy agreement. |
| 83 | Reload .h5 and compare predictions | Completed the truncated HDF5 parity loop and missing aggregate variables; also checks uncertainty-policy agreement. |
| 84 | Evaluate export parity | Completed the truncated parity status display and policy checks; reports failures honestly while allowing the diagnostic ZIP to be created. |
| 85 | Save parity information | Completed the parity JSON writer; records actual format results and explicitly leaves backend parity untested. |
| 86 | Save target CinnaAI backend requirements | Restored actual newlines between requirements. Target-backend versions are documentation only, not a Colab installation command. |
| 87 | Save actual Colab training requirements | Restored requirement-file newlines and records actual KaggleHub/Matplotlib versions used by training. |
| 88 | Calculate important SHA-256 checksums | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 89 | Create metadata.json | Completed the truncated metadata with preprocessing/output/versions/policy/hashes. No backend or deployment success is asserted before testing. |
| 90 | Create backend parity test script | Completed the truncated generated parity script and writer. Supports either export and verifies model/image/tensor hashes, scores, labels and policy decisions. |
| 91 | Create README for Codex | Completed the truncated handoff README, including results, exact input/output, failed-export handling and integration instructions. |
| 92 | Create complete file checksums | Completed the truncated checksum writer, with real newlines and portable relative paths. |
| 93 | Display final handoff package | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 94 | Create final ZIP | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 95 | Download final model package | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 96 | Optional manual prediction test | Guarded the optional interactive upload so top-to-bottom Run all does not block on an unnecessary manual test. |
| 97 | Predict uploaded image | Added the empty-upload guard corresponding to the explicitly optional manual test. |

## Verification performed

- All 97 code cells retain their supplied numbering, exact titles and order, each preceded by its Markdown heading.
- Python syntax checks passed for the 96 Python cells; CELL 1 is the supplied Colab shell command. Generated preprocessing and backend parity scripts also compile.
- A local CPU smoke test executed CELLS 3–94 and the disabled optional CELLS 96–97 with 48 synthetic images, plus a corrupt image and an exact duplicate.
- The smoke environment used TensorFlow 2.16.2 and Keras 3.15.1. Only the test harness substituted offline randomly initialized EfficientNetB0 weights and one training batch/epoch per stage; the delivered notebook retains ImageNet and 25/35 epoch limits.
- Frozen-head training, selective fine-tuning, Stage 1 candidate preservation, validation selection, held-out evaluation, and Grad-CAM for frozen/fine-tuned models passed.
- Both exported formats had zero maximum score difference on the synthetic smoke fixtures. This does not predict parity for the later real ImageNet-trained model.
- The generated backend script ran successfully for both formats; preprocessing pixels, fixture/model hashes, labels and uncertainty decisions matched.
- Both threshold-target outcomes and failed-parity cases were exercised. JSON, requirement-file line breaks, checksum coverage and ZIP integrity were verified.
- Actual Kaggle access, ImageNet download, Colab GPU execution, browser download/upload and full real-dataset training were not exercised locally. They must run in Colab; no disease performance is claimed from synthetic tests.

## Cross-label duplicate correction

The reported Colab conflict for IMG_1668.JPG is handled by excluding every matching file hash from the training/evaluation manifest. Same-label deduplication in CELL 17 must not decide a conflicting label. CELL 71 exports the excluded rows and counts; original dataset files remain available for review.

Targeted checks passed for exclusion of both conflicting copies, blocking unresolved conflicts in CELL 17, same-label deduplication, the no-conflict path and CSV/summary audit export. All 96 Python cells still pass syntax checks; CELL 1 remains a Colab shell command.
